In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

DIR_INPUT  = Path("rider_data")   # fichiers enrichis
DIR_OUTPUT = Path("rider_data")

In [ ]:
BARDET_OUT = Path("rider_data")  # ← adapté

df_b = pd.read_csv(DIR_INPUT / "bardet_romain.csv", low_memory=False)
df_b['date']     = pd.to_datetime(df_b['date'], errors='coerce')
df_b['selected'] = pd.to_numeric(df_b['selected'], errors='coerce')
df_b['distance_gpx_km'] = pd.to_numeric(df_b['distance_gpx_km'], errors='coerce')

df_b = df_b.sort_values('date').reset_index(drop=True)

def compute_load_features(df, window_days=30):
    """
    Pour chaque ligne :
    - n_races_30d   : nb d'étapes courues (selected=1) dans les 30j précédents
    - km_30d        : km totaux parcourus (selected=1) dans les 30j précédents
    Strictement avant la date courante (pas de fuite).
    """
    df = df.sort_values('date').copy()

    # Garder seulement les étapes réellement courues pour le calcul
    raced = df[df['selected'] == 1][['date','distance_gpx_km']].copy()
    raced['distance_gpx_km'] = raced['distance_gpx_km'].fillna(0)

    n_races_list = []
    km_list      = []

    for _, row in df.iterrows():
        if pd.isna(row['date']):
            n_races_list.append(np.nan)
            km_list.append(np.nan)
            continue

        cutoff_start = row['date'] - pd.Timedelta(days=window_days)
        cutoff_end   = row['date']  # strictement avant

        past = raced[
            (raced['date'] >= cutoff_start) &
            (raced['date'] <  cutoff_end)
        ]

        n_races_list.append(len(past))
        km_list.append(past['distance_gpx_km'].sum())

    df['n_races_30d'] = n_races_list
    df['km_30d']      = km_list
    return df

df_b = compute_load_features(df_b)

BARDET_OUT.mkdir(parents=True, exist_ok=True)
df_b.to_csv(BARDET_OUT / "bardet_romain.csv", index=False)  # ← sauvegarde ici
print(f"✅ Bardet sauvegardé → {BARDET_OUT / 'bardet_romain.csv'}")

# ── Vérification ─────────────────────────────────────────────
print("Distribution n_races_30d :")
print(df_b['n_races_30d'].describe().round(1).to_string())

print("\nDistribution km_30d :")
print(df_b['km_30d'].describe().round(1).to_string())

print("\nSpot check 15 lignes :")
print(
    df_b[df_b['selected'] == 1]
    [['date','course','selected','statut','distance_gpx_km',
      'n_races_30d','km_30d']]
    .head(15)
    .to_string(index=False)
)

In [103]:
# Vérifier manuellement un exemple précis
# Prendre une ligne au hasard et recomputer à la main
sample_row = df_b[df_b['selected'] == 1].sample(n=1, random_state=None).iloc[0]

print(f"Ligne vérifiée :")
print(f"  Date     : {sample_row['date'].date()}")
print(f"  Course   : {sample_row['course']}")
print(f"  n_races  : {sample_row['n_races_30d']}")
print(f"  km_30d   : {sample_row['km_30d']:.1f}")

# Recalculer manuellement
cutoff_start = sample_row['date'] - pd.Timedelta(days=30)
past_manual = df_b[
    (df_b['selected'] == 1) &
    (df_b['date'] >= cutoff_start) &
    (df_b['date'] <  sample_row['date'])
][['date','course','distance_gpx_km']]

print(f"\nÉtapes passées (30j) trouvées manuellement : {len(past_manual)}")
print(past_manual.to_string(index=False))
print(f"Total km manuel : {past_manual['distance_gpx_km'].sum():.1f}")
print(f"\n{'✅ OK' if len(past_manual) == sample_row['n_races_30d'] else '❌ ERREUR'}")

Ligne vérifiée :
  Date     : 2024-07-20
  Course   : tour-de-france
  n_races  : 19
  km_30d   : 3337.2

Étapes passées (30j) trouvées manuellement : 19
      date         course  distance_gpx_km
2024-06-29 tour-de-france           206.10
2024-06-30 tour-de-france           201.10
2024-07-01 tour-de-france           231.11
2024-07-02 tour-de-france           139.83
2024-07-03 tour-de-france           177.90
2024-07-04 tour-de-france           163.85
2024-07-05 tour-de-france            25.24
2024-07-06 tour-de-france           183.59
2024-07-07 tour-de-france           198.71
2024-07-09 tour-de-france           187.25
2024-07-10 tour-de-france           211.14
2024-07-11 tour-de-france           203.91
2024-07-12 tour-de-france           165.34
2024-07-13 tour-de-france           152.22
2024-07-14 tour-de-france           198.40
2024-07-16 tour-de-france           188.56
2024-07-17 tour-de-france           178.05
2024-07-18 tour-de-france           179.84
2024-07-19 tour-de-france    

In [ ]:
all_files = list(DIR_INPUT.glob("*.csv"))
print(f"Fichiers à enrichir : {len(all_files)}")

errors = []
for f in tqdm(all_files, desc="Calcul charge 30j"):
    try:
        df = pd.read_csv(f, low_memory=False)
        df['date']           = pd.to_datetime(df['date'], errors='coerce')
        df['selected']       = pd.to_numeric(df['selected'], errors='coerce')
        df['distance_gpx_km'] = pd.to_numeric(df['distance_gpx_km'], errors='coerce')

        # Supprimer si déjà calculé
        df = df.drop(columns=['n_races_30d','km_30d'], errors='ignore')

        df = compute_load_features(df, window_days=30)
        df.to_csv(DIR_OUTPUT / f.name, index=False)

    except Exception as e:
        errors.append((f.name, str(e)))

print(f"\n✅ {len(all_files) - len(errors)} fichiers enrichis → {DIR_OUTPUT}")
if errors:
    print(f"⚠️  {len(errors)} erreurs :")
    for name, err in errors[:10]:
        print(f"   {name} : {err}")

In [ ]:
# Vérification rapide sur quelques fichiers
sample_files = list(DIR_OUTPUT.glob("*.csv"))[:500]
stats = []

for f in sample_files:
    try:
        df = pd.read_csv(f, low_memory=False, usecols=['n_races_30d','km_30d','selected'])
        df_sel = df[pd.to_numeric(df['selected'], errors='coerce') == 1]
        stats.append({
            'n_races_30d_moy' : pd.to_numeric(df_sel['n_races_30d'], errors='coerce').mean(),
            'km_30d_moy'      : pd.to_numeric(df_sel['km_30d'],      errors='coerce').mean(),
        })
    except: pass

df_s = pd.DataFrame(stats)
print("Stats globales (selected=1) :")
print(f"  n_races_30d moyen : {df_s['n_races_30d_moy'].mean():.1f} étapes")
print(f"  km_30d moyen      : {df_s['km_30d_moy'].mean():.0f} km")
print(f"  km_30d max        : {df_s['km_30d_moy'].max():.0f} km")